# Workflow for HTT magspec analysis

This notebook walks through the full HTT magspec analysis pipeline: from raw camera images, through per-camera charge/energy calibration, to a stitched MeV-vs-charge spectrum. Along the way it demonstrates the **`background_source` directive** — the ScanAnalysis-side way to declare "use scan N's data as background for this device" without hardcoding file paths.

In [ ]:
# Boiler plate config/logging setup

import logging
import time
from IPython.display import Image, display

from geecs_data_utils import ScanPaths, ScanTag
from scan_analysis.analyzers.common.array2D_scan_analysis import Array2DScanAnalyzer
from scan_analysis.analyzers.common.array1d_scan_analysis import Array1DScanAnalyzer
from scan_analysis.config.diagnostic_models import BackgroundSource

# Image analyzers used by this notebook
from image_analysis.analyzers.magspec_manual_calib_analyzer import (
    MagSpecManualCalibAnalyzer,
)
from image_analysis.analyzers.line_stitcher import LineStitcher
from image_analysis.config import load_camera_config, load_line_config
from geecs_data_utils.config_roots import image_analysis_config

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logging.getLogger("image_analysis").setLevel(logging.ERROR)
logging.getLogger("scan_analysis").setLevel(logging.ERROR)
logging.getLogger("geecs_data_utils").setLevel(logging.WARNING)
logging.getLogger("logmaker_4_googledocs").setLevel(logging.ERROR)

image_analysis_config.set_base_dir(ScanPaths.paths_config.image_analysis_configs_path)

## Per-camera scan analyzers with background-from-scan-N

Each MagCam camera gets its own `Array2DScanAnalyzer` wrapping a `MagSpecManualCalibAnalyzer`. We attach a `BackgroundSource` directive to each so the scan analyzer computes its own background from the designated scan number — no manual file lookups, no hardcoded paths.

The directive is resolved at runtime by `SingleDeviceScanAnalyzer._resolve_background_paths`, which:
1. Locates the background-scan's data for this device
2. Computes a background frame (median by default)
3. Caches the result to disk
4. Rewrites `camera_config.background` to `method=FROM_FILE, file_path=<cache>` before the per-shot loop runs

In [ ]:
# Scan-tag of the data to analyze
test_tag = ScanTag(year=2026, month=4, day=10, number=14, experiment="Thomson")

# Scan-tag whose shots provide the background frame
bkg_scan_number = 13


def build_magcam_scan_analyzer(config_name, dev_name):
    """Construct an Array2DScanAnalyzer for one MagCam, with bkg-from-scan directive."""
    cam_config = load_camera_config(config_name)
    image_analyzer = MagSpecManualCalibAnalyzer(camera_config=cam_config)
    scan_analyzer = Array2DScanAnalyzer(
        image_analyzer=image_analyzer,
        device_name=dev_name,
        flag_save_images=True,
    )
    # The directive: use scan_number=bkg_scan_number as the background source.
    # SingleDeviceScanAnalyzer picks this up at runtime.
    scan_analyzer.background_source = BackgroundSource(scan_number=bkg_scan_number)
    return scan_analyzer


magcam_pipelines = [
    build_magcam_scan_analyzer("HTT-MagCam1", "HTT-C23_1_MagSpec1"),
    build_magcam_scan_analyzer("HTT-MagCam2", "HTT-C23_2_MagSpec2"),
    build_magcam_scan_analyzer("HTT-MagCam3", "HTT-C23_3_MagSpec3"),
    build_magcam_scan_analyzer("HTT-MagCam4", "HTT-C23_4_MagSpec4"),
]

t0 = time.monotonic()
for p in magcam_pipelines:
    p.run_analysis(scan_tag=test_tag)
print(f"per-camera execution time: {time.monotonic() - t0:.2f}s")

## Stitch per-camera spectra into a single MeV vs charge curve

Once each camera has its charge/energy calibration applied (the previous step), the `LineStitcher` concatenates the four per-camera spectra, sorts by energy, and linearly interpolates onto a uniform x-grid. The `master_device` (MagSpec1) defines the line config and the file-discovery anchor; the three `sibling_devices` are resolved by name-swap against the master's path.

In [ ]:
config_name = "HTT-InterpSpec"
master_device = "HTT-C23_1_MagSpec1-interpSpec"

sibling_devices = [
    "HTT-C23_2_MagSpec2-interpSpec",
    "HTT-C23_3_MagSpec3-interpSpec",
    "HTT-C23_4_MagSpec4-interpSpec",
]

line_config = load_line_config(config_name)
magspec_stitcher = LineStitcher(
    line_config=line_config,
    sibling_devices=sibling_devices,
    name="HTT-C23_MagSpecStitched",
)

scan_analyzer_mag = Array1DScanAnalyzer(
    image_analyzer=magspec_stitcher,
    device_name=master_device,
    file_tail=".tsv",
)

result = scan_analyzer_mag.run_analysis(scan_tag=test_tag)
display(Image(filename=str(result[0]), width=500))